# Step 2 - KLUE-BERT 임베딩 추출 (frozen)

- 입력: 결합 텍스트 (`jdgmn + ' ' + summ_pass`)
- `klue/bert-base`를 **frozen**으로 사용, `[CLS]` 768차원 추출
- train/val/test 임베딩을 `artifacts/`에 `.npy`로 저장 (재추출 방지)
- **이 노트북을 가장 먼저 실행** (bert_mlp, late_fusion이 이 결과를 사용)

In [1]:
import os, time, numpy as np, pandas as pd

# 스모크 테스트: True면 클래스별 소량 샘플만 사용 (파이프라인 점검용)
SMOKE_TEST = False
N_SMOKE = 1000

def find_root(start='.'):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'processed_data')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(start)

ROOT = find_root()
DATA_DIR = os.path.join(ROOT, 'processed_data')
ART_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(ART_DIR, exist_ok=True)
print('ROOT     :', ROOT)
print('DATA_DIR :', DATA_DIR)
print('ART_DIR  :', ART_DIR)

ROOT     : /workspace/big_data_programming-LEON
DATA_DIR : /workspace/big_data_programming-LEON/processed_data
ART_DIR  : /workspace/big_data_programming-LEON/artifacts


In [2]:
# 라벨 순서 (label_mapping.csv 기준: 인덱스 = label 값)
LABEL_NAMES = ['가사', '개인정보/ICT', '근로자', '금융조세', '기업',
               '민사', '특허/저작권', '행정', '형사A(생활형)', '형사B(일반형)']
N_CLASS = 10

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'klue/bert-base'
MAX_LEN = 512
BATCH = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


In [4]:
label_map = pd.read_csv(os.path.join(DATA_DIR, 'label_mapping.csv'))
label_map.columns = label_map.columns.str.strip()
CLASS_TO_LABEL = dict(zip(label_map['class_name'], label_map['label']))

def load_split(name):
    """결합 입력(text = 판시사항 + 요약문)과 라벨 반환."""
    df = pd.read_csv(os.path.join(DATA_DIR, name + '.csv'))
    df.columns = df.columns.str.strip()

    if 'label' not in df.columns:
        if 'class_name' not in df.columns:
            raise KeyError(f"{name}.csv에 label/class_name 컬럼이 없습니다: {df.columns.tolist()}")
        df['label'] = df['class_name'].map(CLASS_TO_LABEL)

    df['label'] = df['label'].astype(int)

    if SMOKE_TEST:
        per = max(2, N_SMOKE // N_CLASS)
        parts = []
        for _, g in df.groupby('label', sort=True):
            parts.append(g.sample(min(len(g), per), random_state=42))
        df = pd.concat(parts, ignore_index=True)
        
    text = (df['jdgmn'].fillna('') + ' ' + df['summ_pass'].fillna('')).tolist()
    labels = df['label'].to_numpy()
    return text, labels

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
for p in model.parameters():
    p.requires_grad = False
print('model loaded, params frozen')

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model loaded, params frozen


In [6]:
@torch.no_grad()
def extract(texts):
    out = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i + BATCH]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors='pt').to(device)
        cls = model(**enc).last_hidden_state[:, 0, :]
        out.append(cls.cpu().numpy())
        if (i // BATCH) % 50 == 0:
            print(f'  {i}/{len(texts)}')
    return np.vstack(out).astype('float32')

In [7]:
for split in ['train', 'val', 'test']:
    texts, labels = load_split(split)
    t0 = time.time()
    emb = extract(texts)
    np.save(os.path.join(ART_DIR, f'bert_{split}_X.npy'), emb)
    np.save(os.path.join(ART_DIR, f'bert_{split}_y.npy'), labels)
    print(f'{split}: X={emb.shape} y={labels.shape}  ({time.time()-t0:.0f}s) saved')
print('done - 임베딩 저장 완료')

  0/34080
  1600/34080
  3200/34080
  4800/34080
  6400/34080
  8000/34080
  9600/34080
  11200/34080
  12800/34080
  14400/34080
  16000/34080
  17600/34080
  19200/34080
  20800/34080
  22400/34080
  24000/34080
  25600/34080
  27200/34080
  28800/34080
  30400/34080
  32000/34080
  33600/34080
train: X=(34080, 768) y=(34080,)  (154s) saved
  0/4260
  1600/4260
  3200/4260
val: X=(4260, 768) y=(4260,)  (19s) saved
  0/4261
  1600/4261
  3200/4261
test: X=(4261, 768) y=(4261,)  (21s) saved
done - 임베딩 저장 완료
